In [ ]:
import os
import logging
import chromadb
from chromadb.config import Settings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from chromadb.utils.embedding_functions import SentenceTransformerEmbeddingFunction

# 1. Configure Production Logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

def _extract_field(text: str, field_name: str) -> str | None:
    """
    Scans each line of the text looking for 'FIELD_NAME: value'.
    Returns the value if found, None if not.
    """
    for line in text.splitlines():
        if line.startswith(f"{field_name}:"):
            value = line.split(":", 1)[1].strip()
            return value if value else None
    return None

class ThreatIntelDB:
    def __init__(self, db_path: str = "./brain"):
        """Initializes the persistent vector database."""
        try:
            self.client = chromadb.PersistentClient(path=db_path)
            embedding_fn = SentenceTransformerEmbeddingFunction(
                model_name="all-MiniLM-L6-v2"
            )

            self.collection = self.client.get_or_create_collection(
                name="mitre_threat_intel",
                embedding_function=embedding_fn,  
                metadata={"hnsw:space": "cosine"} 
            )
            # Initialize the splitter for large documents (CISA reports, etc.)
            self.splitter = RecursiveCharacterTextSplitter(
                chunk_size=1000,
                chunk_overlap=200,
                separators=["\n\n", "\n", " ", ""]
            )
            logger.info(f"Successfully connected to ChromaDB at {db_path}")
        except Exception as e:
            logger.error(f"Failed to initialize ChromaDB: {e}")
            raise

    def process_directory(self, directory_path: str):
        """Iterates through a directory, chunks text, and prepares for ingestion."""
        if not os.path.exists(directory_path):
            logger.error(f"Directory {directory_path} not found.")
            return

        for filename in os.listdir(directory_path):
            if filename.endswith(".txt"):          # ← gate check FIRST
                filepath = os.path.join(directory_path, filename)
                try:                               # ← try sits INSIDE the if
                    with open(filepath, 'r', encoding='utf-8') as f:
                        raw_text = f.read()

                    technique_id = _extract_field(raw_text, "TECHNIQUE_ID") \
                            or _extract_field(raw_text, "VULNERABILITY_ID") \
                            or filename.replace(".txt", "")

                    tactic     = _extract_field(raw_text, "TACTIC") or "unknown"
                    date_added = _extract_field(raw_text, "DATE_ADDED") or "unknown"

                    chunks = self.splitter.create_documents(
                        texts=[raw_text],
                        metadatas=[{
                            "source":       filename,
                            "type":         "threat_intel_report",
                            "technique_id": technique_id,
                            "tactic":       tactic,
                            "date_added":   date_added
                        }]
                    )

                    documents = [c.page_content for c in chunks]
                    metadatas = [c.metadata for c in chunks]
                    ids       = [f"{filename}_{i}" for i in range(len(chunks))]

                    self.collection.upsert(
                        documents=documents,
                        metadatas=metadatas,
                        ids=ids
                    )
                    logger.info(f"Ingested {len(chunks)} chunks from {filename} "
                                f"[id={technique_id}, tactic={tactic}, date={date_added}]")

                except Exception as e:
                    logger.error(f"Error processing {filename}: {e}")  # ← filename, not filepath
    def semantic_search(self, query: str, n_results: int = 3):
        """Retrieves the most semantically similar threat reports."""
        logger.info(f"Executing semantic search for: '{query}'")
        try:
            results = self.collection.query(
                query_texts=[query],
                n_results=n_results
            )
            return results
        except Exception as e:
            logger.error(f"Search failed: {e}")
            return None

# --- Execution Block ---
if __name__ == "__main__":
    db = ThreatIntelDB()

    intel_dir = "./threat_reports"

    if not os.path.exists(intel_dir):
        os.makedirs(intel_dir)
        with open(f"{intel_dir}/sample_mitre.txt", "w") as f:
            f.write("T1566: Phishing. Adversaries may send phishing messages to gain access. "
                    "This often involves malicious links or attachments.")

    db.process_directory(intel_dir)

    question = "How do hackers trick employees into clicking bad links to get inside the network?"
    search_results = db.semantic_search(question)

    if search_results and search_results['documents'][0]:
        logger.info("--- RAG RETRIEVAL RESULTS ---")
        for i in range(len(search_results['documents'][0])):
            doc  = search_results['documents'][0][i]
            meta = search_results['metadatas'][0][i]
            logger.info(f"Source: {meta['source']}")
            logger.info(f"Technique ID: {meta.get('technique_id', 'N/A')}")
            logger.info(f"Tactic: {meta.get('tactic', 'N/A')}")
            logger.info(f"Content: {doc[:300]}...")
    else:
        logger.warning("No results found. Check if documents were ingested correctly.")